## Data Modeling

In [1]:
import numpy as np, pandas as pd, time
from pathlib import Path
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
P = ROOT / "data" / "processed"
print(P, P.exists())          # should print True


X_train = sparse.load_npz(P / "X_train.npz").toarray()
y_train = np.load(P / "y_train.npy")
names = pd.read_csv(P / "feature_names.csv")["feature"]

print(X_train.shape, y_train.shape, len(names))

/content/data/processed False


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/processed/X_train.npz'

In [ ]:
models = {
    "Baseline (no skill)": DummyClassifier(strategy="prior"),
    "Logistic regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Decision tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=200, class_weight='balanced',random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=50, class_weight='balanced',n_jobs=-1, random_state=42),
    "Gradient boosting": HistGradientBoostingClassifier(class_weight='balanced', max_iter=200, random_state=42)
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for name, m in models.items():
    t = time.time()
    r = cross_validate(m,X_train, y_train, cv=cv, scoring={"roc_auc":"roc_auc","pr_auc":"average_precision"})
    rows.append({"model":name, "roc_auc":r["test_roc_auc"].mean(), "roc_sd":r["test_roc_auc"].std(),
                 "pr_auc": r["test_pr_auc"].mean(), "minutes":(time.time() - t) /60})
results = pd.DataFrame(rows).set_index("model").round(3).sort_values("pr_auc", ascending=False)
results

KeyboardInterrupt: 